## Instalación de dependencias

In [ ]:
!pip install pandas matplotlib seaborn wordcloud numpy

## Imports y carga del corpus

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter
import re

# ── Estilo global de gráficos ─────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 130,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "font.family": "sans-serif",
})

AZUL      = "#2E5496"
AZUL_CLAR = "#A8C4E0"
ROJO      = "#C0392B"
VERDE     = "#27AE60"
GRIS      = "#7F8C8D"

# ── Carga del corpus ──────────────────────────────────────────────────────────
df = pd.read_csv("corpus_tweets_final.csv", parse_dates=False)
df["año_mes"] = df["año_mes"].astype(str)
df["año"]     = df["año"].astype(int)
df["mes"]     = df["mes"].astype(int)

print(f"Corpus cargado: {len(df):,} tweets")
print(f"Período: {df['año_mes'].min()} → {df['año_mes'].max()}")
print(f"Columnas: {list(df.columns)}")


---
## Volumen temporal

### Tweets por mes




In [ ]:
# ── Datos ─────────────────────────────────────────────────────────────────────
vol = df["año_mes"].value_counts().sort_index()
meses  = list(vol.index)
valores = list(vol.values)

# Color por año
COLORES_AÑO = {"2023": "#1a5276", "2024": "#2980b9", "2025": "#85c1e9", "2026": "#e74c3c"}
colores_barras = [COLORES_AÑO[m[:4]] for m in meses]

# Hitos académicos
HITOS = {
    "2023-06": "Conv. extra.", "2023-09": "Inicio 23/24",
    "2024-02": "Exám. 1º C.", "2024-06": "Conv. extra.",
    "2024-09": "Inicio 24/25","2025-02": "Exám. 1º C.",
    "2025-06": "Conv. extra.", "2025-09": "Inicio 25/26",
    "2026-02": "Exám. 1º C.",
}

fig, ax = plt.subplots(figsize=(16, 6), facecolor="white")
ax.set_facecolor("white")

bars = ax.bar(range(len(meses)), valores, color=colores_barras,
              width=0.75, zorder=3, edgecolor="white", linewidth=0.4)

# Media móvil
media_movil = pd.Series(valores).rolling(3, center=True).mean()
ax.plot(range(len(meses)), media_movil, color="#555555", linewidth=1.8,
        linestyle="--", zorder=4, label="Media móvil (3 meses)")

# Media global
media_global = sum(valores)/len(valores)
ax.axhline(media_global, color="#AAAAAA", linewidth=1.2, linestyle=":",
           zorder=2, label=f"Media global ({media_global:.0f} tw/mes)")

# Anotaciones hitos
for mes_label, texto in HITOS.items():
    if mes_label in meses:
        idx = meses.index(mes_label)
        yval = valores[idx]
        ax.annotate(texto, xy=(idx, yval),
                    xytext=(idx, yval + 14), fontsize=7.5, color="#333333",
                    ha="center", va="bottom",
                    arrowprops=dict(arrowstyle="-", color="#AAAAAA", lw=0.8))

# Nota explicativa para 2026-03 (último mes incompleto)
idx_last = len(meses) - 1
ax.annotate("* Mes en curso\nal cierre de\nla recogida",
            xy=(idx_last, valores[idx_last]),
            xytext=(idx_last - 3, valores[idx_last] + 20),
            fontsize=8, color="#e74c3c", ha="center",
            arrowprops=dict(arrowstyle="->", color="#e74c3c", lw=1))

# Eje X: etiquetas año-mes cada 3 meses
etiquetas_x = []
for m in meses:
    año, mes_n = m.split("-")
    if mes_n in ("01", "04", "07", "10"):
        etiquetas_x.append(f"{mes_n}/{año[2:]}")
    else:
        etiquetas_x.append("")

ax.set_xticks(range(len(meses)))
ax.set_xticklabels(etiquetas_x, rotation=45, ha="right", fontsize=8.5, color="#222222")

# Leyenda de años
from matplotlib.patches import Patch
patches_años = [Patch(facecolor=c, label=f"Año {a}")
                for a, c in COLORES_AÑO.items()]
lineas = [
    plt.Line2D([0],[0], color="#555555", lw=1.8, linestyle="--", label="Media móvil"),
    plt.Line2D([0],[0], color="#AAAAAA", lw=1.2, linestyle=":", label=f"Media global ({media_global:.0f})"),
]
ax.legend(handles=patches_años + lineas, fontsize=8.5, loc="upper left",
          frameon=True, facecolor="white", edgecolor="#CCCCCC",
          ncol=2, labelcolor="#222222")

ax.set_ylabel("Número de tweets", color="#222222", fontsize=10)
ax.set_xlabel("Mes / Año", color="#222222", fontsize=10)
ax.set_title("Fig. 1 — Volumen mensual de tweets sobre la UAM Madrid (2023–2026)",
             fontsize=12, fontweight="bold", color="#222222", pad=10)
ax.tick_params(colors="#222222")
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
ax.spines["left"].set_color("#DDDDDD"); ax.spines["bottom"].set_color("#DDDDDD")
ax.grid(axis="y", alpha=0.25, zorder=1)
ax.set_ylim(0, max(valores) * 1.35)

plt.tight_layout()
plt.savefig("fig1_volumen_mensual.png", bbox_inches="tight", facecolor="white", dpi=150)
plt.show()

print(f"Mínimo:  {vol.min()} tweets ({vol.idxmin()})")
print(f"Máximo:  {vol.max()} tweets ({vol.idxmax()})  ← mes en curso al cierre")
print(f"Media:   {media_global:.1f} tweets/mes")
print(f"Mediana: {vol.median():.0f} tweets/mes")


### Estacionalidad: tweets por mes del año

.


In [ ]:
vol_mes = df.groupby("mes").size()
nombres_mes = ["Ene","Feb","Mar","Abr","May","Jun",
               "Jul","Ago","Sep","Oct","Nov","Dic"]
media_est = vol_mes.mean()

# Etiquetas de contexto académico por mes
CONTEXTO = {
    1: "Exám.\nfin 1C", 2: "Exám.\nfin 1C", 3: "Exám.\nparciales",
    6: "Conv.\nextra.", 7: "Exám.\nfin 2C", 9: "Inicio\ncurso",
    11: "Exám.\nparciales"
}

fig, ax = plt.subplots(figsize=(12, 5), facecolor="white")
ax.set_facecolor("white")

colores_est = ["#e74c3c" if v >= media_est else "#2980b9" for v in vol_mes.values]
bars = ax.bar(range(1, 13), vol_mes.values, color=colores_est,
              width=0.65, zorder=3, edgecolor="white")

ax.axhline(media_est, color="#888888", linewidth=1.3, linestyle="--",
           zorder=2, label=f"Media mensual acumulada ({media_est:.0f} tweets)")

# Valores encima de cada barra
for i, (m, v) in enumerate(zip(range(1, 13), vol_mes.values)):
    ax.text(m, v + 4, str(v), ha="center", va="bottom",
            fontsize=9, color="#222222", fontweight="bold")
    # Contexto académico debajo del eje
    if m in CONTEXTO:
        ax.text(m, -28, CONTEXTO[m], ha="center", va="top",
                fontsize=7.5, color="#555555", style="italic")

ax.set_xticks(range(1, 13))
ax.set_xticklabels(nombres_mes, fontsize=10, color="#222222")
ax.set_xlabel("Mes del año (acumulado 2023–2026)", color="#222222", fontsize=10)
ax.set_ylabel("Tweets acumulados", color="#222222", fontsize=10)
ax.set_title("Fig. 2 — Estacionalidad: actividad acumulada por mes del año\n"
             "(en rojo: meses por encima de la media; contexto académico en cursiva)",
             fontsize=11, fontweight="bold", color="#222222", pad=8)
ax.tick_params(colors="#222222")
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
ax.spines["left"].set_color("#DDDDDD"); ax.spines["bottom"].set_color("#DDDDDD")
ax.grid(axis="y", alpha=0.25, zorder=1)
ax.set_ylim(-45, max(vol_mes.values) * 1.18)

from matplotlib.patches import Patch
leyenda = [
    Patch(facecolor="#e74c3c", label="Por encima de la media (alta actividad)"),
    Patch(facecolor="#2980b9", label="Por debajo de la media"),
    plt.Line2D([0],[0], color="#888888", lw=1.3, linestyle="--",
               label=f"Media acumulada ({media_est:.0f} tweets/mes)"),
]
ax.legend(handles=leyenda, fontsize=9, loc="upper right",
          frameon=True, facecolor="white", edgecolor="#CCCCCC", labelcolor="#222222")

plt.tight_layout()
plt.savefig("fig2_estacionalidad.png", bbox_inches="tight", facecolor="white", dpi=150)
plt.show()


---
## Composición del corpus

### Distribución por tipo de query



In [ ]:
QUERY_INFO = {
    "mention_query":               ("v1", "Mención @UAM_Madrid"),
    "university_name":             ("v1", "Nombre completo UAM"),
    "student_problems":            ("v1", "Problemas estudiantiles"),
    "experiencia_directa":         ("v2", "Experiencia directa"),
    "valoracion_explicita":        ("v2", "Valoración explícita"),
    "momentos_academicos":         ("v2", "Momentos académicos"),
    "experiencia_uam_madrid":      ("v3", "Experiencia UAM Madrid"),
    "valoracion_uam_madrid":       ("v3", "Valoración UAM Madrid"),
    "momentos_academicos_madrid":  ("v3", "Momentos acad. Madrid"),
}
COLORES_VERSION = {"v1": "#1a5276", "v2": "#e67e22", "v3": "#27ae60"}

qt = df["query_type"].value_counts()
etiquetas = [QUERY_INFO.get(q, ("??", q))[1] for q in qt.index]
versiones  = [QUERY_INFO.get(q, ("v1", q))[0] for q in qt.index]
colores_qt = [COLORES_VERSION[v] for v in versiones]

fig, axes = plt.subplots(1, 2, figsize=(16, 6), facecolor="white")

# ── Panel izquierdo ───────────────────────────────────────────────────────────
ax = axes[0]
ax.set_facecolor("white")
bars = ax.barh(range(len(qt)), qt.values, color=colores_qt,
               height=0.6, zorder=3, edgecolor="white")
ax.set_yticks(range(len(qt)))
ax.set_yticklabels(etiquetas, fontsize=10.5, color="#111111")
ax.invert_yaxis()
ax.set_xlabel("Número de tweets", color="#222222", fontsize=10)
ax.set_title("Tweets por tipo de query", color="#222222",
             fontsize=12, fontweight="bold")
ax.tick_params(colors="#222222")
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
ax.spines["left"].set_color("#DDDDDD"); ax.spines["bottom"].set_color("#DDDDDD")
ax.grid(axis="x", alpha=0.25, zorder=1)
ax.set_xlim(0, max(qt.values) * 1.18)
for i, v in enumerate(qt.values):
    ax.text(v + 4, i, str(v), va="center", fontsize=9.5, color="#222222")

# Leyenda versiones debajo del gráfico
from matplotlib.patches import Patch
leyenda_v = [
    Patch(facecolor="#1a5276", label="v1 — Recogida inicial (queries amplias)"),
    Patch(facecolor="#e67e22", label="v2 — Queries orientadas a estudiantes"),
    Patch(facecolor="#27ae60", label="v3 — Queries con anclaje geográfico"),
]
ax.legend(handles=leyenda_v, fontsize=9, loc="lower right",
          frameon=True, facecolor="white", edgecolor="#CCCCCC", labelcolor="#222222")

# ── Panel derecho: tarta ──────────────────────────────────────────────────────
ax2 = axes[1]
ax2.set_facecolor("white")
ver_counts = df["version"].value_counts().reindex(["v1","v2","v3"])
colores_pie = [COLORES_VERSION[v] for v in ver_counts.index]

wedges, texts, autotexts = ax2.pie(
    ver_counts.values,
    labels=[f"Recogida {v}\n({n:,} tweets)" for v, n in ver_counts.items()],
    colors=colores_pie,
    autopct="%1.1f%%", startangle=140,
    textprops={"fontsize": 10.5, "color": "#111111"},
    wedgeprops={"edgecolor": "white", "linewidth": 2.5},
    pctdistance=0.65,
)
for at in autotexts:
    at.set_color("white"); at.set_fontsize(11); at.set_fontweight("bold")

ax2.set_title("Distribución por versión de recogida",
              color="#222222", fontsize=12, fontweight="bold")

fig.suptitle("Fig. 3 — Composición del corpus por tipo de query y versión de recogida",
             fontsize=12, fontweight="bold", color="#222222", y=1.01)
plt.tight_layout(pad=2.5)
plt.savefig("fig3_composicion_corpus.png", bbox_inches="tight", facecolor="white", dpi=150)
plt.show()


---
## Análisis de engagement



In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5), facecolor="white")
fig.suptitle("Fig. 4 — Distribución de métricas de engagement\n"
             "(escala logarítmica; cada barra representa un rango de valores)",
             fontsize=12, fontweight="bold", color="#222222", y=1.03)

metricas = [
    ("likes",    "Likes",          AZUL,  "Número de likes recibidos"),
    ("retweets", "Retweets",       VERDE, "Número de veces compartido"),
    ("views",    "Visualizaciones",ROJO,  "Número de veces visto"),
]

for ax, (col, label, color, desc) in zip(axes, metricas):
    ax.set_facecolor("white")
    datos = df[col][df[col] > 0]
    ax.hist(np.log10(datos + 1), bins=28, color=color, alpha=0.85,
            edgecolor="white", zorder=3)

    mediana = datos.median()
    ax.axvline(np.log10(mediana + 1), color="#222222", linewidth=1.8,
               linestyle="--", label=f"Mediana: {mediana:.0f}")

    # Etiquetas en el eje X con valores reales
    ticks_log = [0, 1, 2, 3, 4, 5]
    ax.set_xticks(ticks_log)
    ax.set_xticklabels([f"{int(10**t):,}" for t in ticks_log],
                       fontsize=8, color="#222222")

    ax.set_xlabel("Valor real (escala log₁₀)", color="#222222", fontsize=9)
    ax.set_ylabel("Nº de tweets", color="#222222", fontsize=9)
    ax.set_title(f"{label}\n({desc})", fontsize=10,
                 fontweight="bold", color="#222222")
    ax.tick_params(colors="#222222")
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("#DDDDDD"); ax.spines["bottom"].set_color("#DDDDDD")
    ax.grid(axis="y", alpha=0.25, zorder=1)
    ax.legend(fontsize=9, frameon=True, facecolor="white",
              edgecolor="#CCCCCC", labelcolor="#222222")

    # Anotación % con 0
    pct_0 = (df[col] == 0).mean() * 100
    ax.text(0.97, 0.97, f"{pct_0:.0f}% con valor 0\n(no incluidos)",
            transform=ax.transAxes, fontsize=7.5, color="#666666",
            ha="right", va="top", style="italic")

plt.tight_layout()
plt.savefig("fig4_engagement.png", bbox_inches="tight", facecolor="white", dpi=150)
plt.show()


### Nota sobre outliers de engagement



---
## Vocabulario más frecuente

### Top 30 palabras




In [ ]:
import unicodedata

def quitar_tildes(texto):
    return ''.join(
        c for c in unicodedata.normalize('NFD', str(texto))
        if unicodedata.category(c) != 'Mn'
    )

# Stopwords sin tildes (porque el texto también se normalizará antes del conteo)
STOPWORDS = {
    # Artículos, preposiciones, conjunciones
    'de','la','el','en','que','y','a','los','las','por','con','se','una','un',
    'es','al','del','lo','su','para','como','mas','pero','sus','le','ya','o',
    'este','ha','me','hay','sobre','fue','ser','son','muy','no','mi','te','han',
    'he','tu','yo','nos','les','has','era','ni','cuando','porque','donde','asi',
    'entre','hasta','desde','tras','cada','todo','esta','si','tan','solo','bien',
    'todos','todas','aqui','ahi','hoy','vez','sido','tiene','parte','otros',
    'menos','otro','mismo','dicho','nuestro','nuestra','ellos','ellas','cual',
    'ante','bajo','gran','algo','algun','alguna','algunos','algunas',
    # UAM y Madrid
    'uam','madrid','autonoma','universidad','uam_madrid','cantoblanco',
    # Verbos comodín
    'hace','hacer','hecho','tener','tengo','tenia','puede','poder','podria',
    'quiero','quiere','seria','decir','dijo','dice','siendo','estan','somos',
    'vamos','viene','trata','lleva','sigue','poner','pone','pasa','parece',
    'creo','cree','mayor','nueva','nuevo','buena','bueno','primer','primera',
    'segundo','segunda','dado','dar','https','tambien','traves','ademas',
    'segun','despues','todavia','aun','quien','cuanto','cuantos','cuales',
    # Palabras frecuentes pero no informativas para SA/LDA
    'gracias','enhorabuena','tiempo','anos','anos',
}

words = []
for t in df["text_clean"]:
    t_norm = quitar_tildes(t)   # "está" → "esta", "cómo" → "como"
    words += [w for w in re.findall(r'\b[a-zn]{4,}\b', t_norm)
              if w not in STOPWORDS]

freq = Counter(words)
top30 = freq.most_common(30)
palabras, conteos = zip(*top30)

fig, ax = plt.subplots(figsize=(13, 9), facecolor="white")
ax.set_facecolor("white")

media_c = sum(conteos) / len(conteos)
colores_bar = ["#1a5276" if c > media_c else "#85c1e9" for c in conteos]

bars = ax.barh(range(len(palabras)), conteos, color=colores_bar,
               height=0.72, zorder=3, edgecolor="white")
ax.set_yticks(range(len(palabras)))
ax.set_yticklabels(palabras, fontsize=11, color="#111111")
ax.invert_yaxis()
ax.set_xlabel("Frecuencia absoluta (nº de apariciones en el corpus)",
              color="#222222", fontsize=10)
ax.set_title("Fig. 5 — Top 30 palabras más frecuentes en el corpus\n"
             "(excluidas stopwords, UAM y Madrid)",
             fontsize=12, fontweight="bold", color="#222222", pad=10)
ax.tick_params(colors="#222222")
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
ax.spines["left"].set_color("#DDDDDD"); ax.spines["bottom"].set_color("#DDDDDD")
ax.grid(axis="x", alpha=0.25, zorder=1)
ax.set_xlim(0, max(conteos) * 1.14)

for i, v in enumerate(conteos):
    ax.text(v + 3, i, str(v), va="center", fontsize=9.5, color="#222222")

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor="#1a5276", label="Por encima de la media"),
    Patch(facecolor="#85c1e9", label="Por debajo de la media"),
], fontsize=9, loc="lower right", frameon=True,
   facecolor="white", edgecolor="#CCCCCC", labelcolor="#222222")

plt.tight_layout()
plt.savefig("fig5_top30_palabras.png", bbox_inches="tight", facecolor="white", dpi=150)
plt.show()


### Nube de palabras

In [ ]:
freq_wc = {w: c for w, c in freq.most_common(150) if w not in STOPWORDS}

wc = WordCloud(
    width=1400, height=600,
    background_color="white",
    colormap="Blues",
    max_words=120,
    prefer_horizontal=0.85,
    collocations=False,
    min_font_size=10,
).generate_from_frequencies(freq_wc)

fig, ax = plt.subplots(figsize=(14, 6))
ax.imshow(wc, interpolation="bilinear")
ax.axis("off")
ax.set_title("Nube de palabras — Corpus tweets UAM Madrid (2023–2026)",
             fontsize=13, fontweight="bold", pad=12)
plt.tight_layout()
plt.savefig("fig6_wordcloud.png", bbox_inches="tight")
plt.show()


### Vocabulario por categoría de query


In [ ]:
df["categoria"] = df["query_type"].map({
    "mention_query":              "Mencion institucional",
    "university_name":            "Mencion institucional",
    "student_problems":           "Momentos academicos",
    "momentos_academicos":        "Momentos academicos",
    "momentos_academicos_madrid": "Momentos academicos",
    "experiencia_directa":        "Experiencia y valoracion",
    "experiencia_uam_madrid":     "Experiencia y valoracion",
    "valoracion_explicita":       "Experiencia y valoracion",
    "valoracion_uam_madrid":      "Experiencia y valoracion",
})

categorias    = ["Mencion institucional", "Momentos academicos", "Experiencia y valoracion"]
titulos_cat   = ["Mención institucional", "Momentos académicos", "Experiencia y valoración"]
colores_cat   = ["#1a5276", "#27ae60", "#c0392b"]
descripciones = [
    "Tweets que mencionan la UAM\npor nombre o cuenta oficial",
    "Tweets sobre exámenes,\nmatrículas, TFG/TFM y becas",
    "Tweets con opinión directa\ny valoraciones personales",
]

fig, axes = plt.subplots(1, 3, figsize=(17, 7), facecolor="white")
fig.suptitle("Fig. 7 — Top 15 palabras más frecuentes por categoría semántica",
             fontsize=13, fontweight="bold", color="#222222", y=1.01)

for ax, cat, titulo, color, desc in zip(axes, categorias, titulos_cat, colores_cat, descripciones):
    ax.set_facecolor("white")
    textos = df[df["categoria"] == cat]["text_clean"]
    ws = []
    for t in textos:
        t_norm = quitar_tildes(t)
        ws += [w for w in re.findall(r'\b[a-zn]{4,}\b', t_norm)
               if w not in STOPWORDS]
    top = Counter(ws).most_common(15)
    if not top:
        continue
    pals, cnts = zip(*top)

    ax.barh(range(len(pals)), cnts, color=color,
            alpha=0.85, height=0.65, zorder=3, edgecolor="white")
    ax.set_yticks(range(len(pals)))
    ax.set_yticklabels(pals, fontsize=10.5, color="#111111")
    ax.invert_yaxis()
    ax.set_title(f"{titulo}\n{desc}", fontsize=10,
                 fontweight="bold", color="#222222", pad=6)
    ax.set_xlabel("Frecuencia", color="#222222", fontsize=9)
    ax.tick_params(colors="#222222")
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("#DDDDDD"); ax.spines["bottom"].set_color("#DDDDDD")
    ax.grid(axis="x", alpha=0.25, zorder=1)
    ax.set_xlim(0, max(cnts) * 1.18)
    for i, v in enumerate(cnts):
        ax.text(v + 1, i, str(v), va="center", fontsize=8.5, color="#222222")
    n_cat = len(textos)
    ax.text(0.97, 0.02, f"n = {n_cat:,} tweets", transform=ax.transAxes,
            fontsize=8.5, color="#666666", ha="right", va="bottom", style="italic")

plt.tight_layout(pad=2)
plt.savefig("fig7_vocabulario_categoria.png", bbox_inches="tight",
            facecolor="white", dpi=150)
plt.show()


---
## Resumen de hallazgos descriptivos




In [ ]:
print("=" * 60)
print("RESUMEN DEL ANÁLISIS DESCRIPTIVO")
print("=" * 60)

vol = df["año_mes"].value_counts().sort_index()
vol_mes = df.groupby("mes").size()

print(f"\n1. CORPUS")
print(f"   Total tweets:      {len(df):,}")
print(f"   Período cubierto:  {df['año_mes'].min()} → {df['año_mes'].max()}")
print(f"   Meses cubiertos:   {df['año_mes'].nunique()}")

print(f"\n2. VOLUMEN TEMPORAL")
print(f"   Media mensual:     {vol.mean():.1f} tweets/mes")
print(f"   Mes más activo:    {vol.idxmax()} ({vol.max()} tweets)")
print(f"   Mes menos activo:  {vol.idxmin()} ({vol.min()} tweets)")
meses_pico = [m for m in ["sep","oct","nov","ene","feb","mar"]
              if vol_mes.get(["ene","feb","mar","abr","may","jun",
                              "jul","ago","sep","oct","nov","dic"].index(m)+1, 0) > vol_mes.mean()]
print(f"   Meses de alta actividad: sep, feb, mar, nov (coincide con calendario académico)")

print(f"\n3. COMPOSICIÓN")
for v, n in df["version"].value_counts().reindex(["v1","v2","v3"]).items():
    print(f"   Recogida {v}: {n:,} tweets ({n/len(df)*100:.1f}%)")

print(f"\n4. ENGAGEMENT")
print(f"   Tweets con 0 likes:    {(df['likes']==0).mean()*100:.1f}%")
print(f"   Mediana likes:         {df['likes'].median():.0f}")
print(f"   Mediana retweets:      {df['retweets'].median():.0f}")
print(f"   Mediana views:         {df['views'].median():.0f}")

print(f"\n5. VOCABULARIO (top 10 palabras)")
words2 = []
for t in df["text_clean"]:
    words2 += [w for w in re.findall(r'\b[a-záéíóúüñ]{4,}\b', str(t))
               if w not in STOPWORDS]
top10 = Counter(words2).most_common(10)
for w, c in top10:
    print(f"   {w}: {c}")
